# Teardown / Uninstall — Gateway Monitor
**Label: [NET-NEW] [Unverified — requires Fabric tenant]**

Cleanly removes everything the deploy created, so the tool has a reversible
install (a deployable product needs a safe uninstall path — fuam-basic ships one too).

> **DESTRUCTIVE.** This deletes the lakehouse (and its Delta data), notebooks,
> pipeline, eventhouse/KQL DB, semantic model, report, and Activator rules in the
> target workspace. It does NOT touch the gateway host, the collectors' scheduled
> tasks, or Workspace Monitoring (turn those off separately — see final cell).
> Requires confirmation. Review the item list before running the delete cell.

In [ ]:
# --- Parameters -------------------------------------------------------------
workspace_id   = ""          # target workspace GUID to clean up
dry_run        = True        # True = only LIST what would be deleted (safe default)
item_name_prefix = "GatewayMon"  # only touch items with this prefix (guardrail)
# Item types this tool creates:
managed_types = ["Report","SemanticModel","DataPipeline","Notebook",
                 "KQLQueryset","KQLDatabase","Eventhouse","Lakehouse","Reflex"]


In [ ]:
# --- Discover items ---------------------------------------------------------
# [Unverified] Prefer sempy.fabric inside a Fabric notebook; confirm method names.
import sempy.fabric as fabric
items = fabric.list_items(workspace=workspace_id)
targets = items[
    items["Type"].isin(managed_types) &
    items["Display Name"].str.startswith(item_name_prefix)
]
print(f"{len(targets)} item(s) matched prefix '{item_name_prefix}':")
for _, r in targets.iterrows():
    print(f"  - {r['Display Name']}  ({r['Type']})  {r['Id']}")


In [ ]:
# --- Delete (ordered: dependents before stores) -----------------------------
# Order matters: Report -> SemanticModel -> Pipeline/Notebook -> KQL/Eventhouse -> Lakehouse.
delete_order = ["Report","SemanticModel","DataPipeline","Notebook",
                "KQLQueryset","KQLDatabase","Eventhouse","Reflex","Lakehouse"]

if dry_run:
    print("DRY RUN — nothing deleted. Set dry_run=False to execute.")
else:
    for t in delete_order:
        for _, r in targets[targets["Type"] == t].iterrows():
            try:
                fabric.delete_item(workspace=workspace_id, item=r["Id"])  # [Unverified] name
                print(f"deleted {r['Display Name']} ({t})")
            except Exception as e:
                print(f"[skip] {r['Display Name']} ({t}): {e}")
    print("Teardown of Fabric items complete.")


## Manual steps NOT done by this notebook
1. **Gateway host** — remove the scheduled tasks / collector scripts on each node
   (they only write to the lakehouse; harmless once the lakehouse is gone, but clean up).
2. **Workspace Monitoring** — turn off in *Workspace settings → Monitoring* if you
   enabled it solely for this tool (it has its own Eventhouse + cost).
3. **Service principal / Key Vault** — revoke the SP's gateway Admin role and delete
   the Key Vault secrets if they were created only for this tool.
4. **FPM bridge** — if you shortcutted FPM's Eventhouse, remove that shortcut.